# Autopsia del trote: la policy bajo el microscopio

Analiza una grabación de `03_walk_policy.py` caminando (hecha con `scripts/02_record_lowstate.py`).

**Para generar datos** (con el sim corriendo):
```bash
# terminal A                                    # terminal B (arrancar ~2 s después)
python scripts/03_walk_policy.py --cmd 0.5 0 0 --dur 14
                                                python scripts/02_record_lowstate.py --dur 16
```

Qué vamos a buscar en las señales:
1. **El patrón de trote**: pares diagonales de patas moviéndose en fase (FR+RL vs FL+RR).
2. **La frecuencia de zancada**: cuántos pasos por segundo eligió dar la red.
3. **El temblor**: la firma del sim2sim gap — energía de alta frecuencia que no debería estar.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

rec_file = sorted(Path("../data").glob("lowstate_*.npz"))[-1]  # la más reciente
d = np.load(rec_file)
t, q, dq, gyro = d["t"], d["q"], d["dq"], d["gyro"]
hz = len(t) / (t[-1] - t[0])
print(f"{rec_file.name}: {len(t)} muestras, ~{hz:.0f} Hz, {t[-1]:.1f} s")

LEGS = ["FR", "FL", "RR", "RL"]
THIGH = [1, 4, 7, 10]  # índice del muslo de cada pata en el orden lowstate

## 1. El patrón de trote

Graficamos el ángulo del **muslo** de las 4 patas. Arriba: la corrida completa (se ve la rampa
inicial, la caminata, y el aflojado final). Abajo: zoom de 3 segundos en plena marcha.

**Qué mirar en el zoom**: si la red trota de verdad, las curvas van de a pares: FR se mueve
junto con RL (diagonal), y FL junto con RR — dos "familias" de ondas en contrafase.
Cada oscilación completa de una curva = una zancada de esa pata.

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 6))
colors = {"FR": "tab:red", "RL": "tab:orange", "FL": "tab:blue", "RR": "tab:cyan"}
# diagonales pintadas en colores 'de familia': cálidos vs fríos
for leg, idx in zip(LEGS, THIGH):
    a1.plot(t, q[:, idx], label=leg, color=colors[leg], lw=0.8)
a1.set_title("q muslo, corrida completa [rad]"); a1.legend(ncol=4); a1.set_xlabel("t [s]")

zoom = (t > 6) & (t < 9)
for leg, idx in zip(LEGS, THIGH):
    a2.plot(t[zoom], q[zoom, idx], label=leg, color=colors[leg], lw=1.5)
a2.set_title("zoom 6–9 s: ¿se ven los pares diagonales FR+RL (cálidos) vs FL+RR (fríos)?")
a2.set_xlabel("t [s]")
fig.tight_layout()

## 2. La frecuencia de zancada

Espectro (FFT) del muslo FR durante la marcha. El pico dominante es el ritmo del trote:
los cuadrúpedos trotan típicamente a 1.5–3 zancadas por segundo. Los picos en múltiplos
(armónicos) son la forma no-senoidal de la zancada — normal. Lo que NO debería haber:
energía apreciable arriba de ~10 Hz (eso ya no es caminar, es vibrar).

In [ ]:
walk = (t > 5) & (t < 13)  # ventana de marcha estable (ajustar si tu corrida difiere)
sig = q[walk, THIGH[0]] - q[walk, THIGH[0]].mean()
freqs = np.fft.rfftfreq(len(sig), d=1 / hz)
power = np.abs(np.fft.rfft(sig)) ** 2

plt.figure(figsize=(11, 3.5))
plt.semilogy(freqs, power, lw=0.9)
plt.xlim(0, 30); plt.xlabel("frecuencia [Hz]"); plt.ylabel("potencia")
peak = freqs[1:][np.argmax(power[1:])]
plt.axvline(peak, color="tab:red", ls="--", label=f"pico: {peak:.1f} Hz")
plt.title("espectro del muslo FR durante la marcha"); plt.legend();
print(f"Frecuencia de zancada dominante: {peak:.1f} Hz (~{peak:.1f} zancadas/s)")

## 3. El temblor, cuantificado

Dos vistas de la "firma nerviosa" de la policy:

- **Velocidad articular (`dq`) en crudo**: en un trote limpio se ven ondas suaves;
  el temblor aparece como pasto de alta frecuencia montado encima.
- **Giróscopo**: cuánto se sacude el *cuerpo*. Comparamos la fase quieta-parada
  (si la hay) con la marcha: un buen controlador parado casi no mueve el gyro.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.5))
w1 = (t > 6) & (t < 8)
a1.plot(t[w1], dq[w1, 2], lw=0.8, label="FR_calf")
a1.plot(t[w1], dq[w1, THIGH[0]], lw=0.8, label="FR_thigh")
a1.set_title("dq en marcha [rad/s]: ¿ondas suaves o pasto?"); a1.legend(); a1.set_xlabel("t [s]")

for i, name in enumerate(["roll", "pitch", "yaw"]):
    a2.plot(t, gyro[:, i], lw=0.7, label=name)
a2.set_title("giróscopo, corrida completa [rad/s]"); a2.legend(); a2.set_xlabel("t [s]")
fig.tight_layout()

print(f"std del giróscopo en marcha: {gyro[walk].std(axis=0).round(2)} rad/s (roll, pitch, yaw)")
print(f"|dq| máximo en marcha: {np.abs(dq[walk]).max():.1f} rad/s")

## Qué significa lo que viste

- Si en el zoom del punto 1 los pares diagonales se distinguen: la red **sí** aprendió el
  trote — el patrón sobrevivió al cambio de simulador. Lo que se degrada es la *calidad*.
- El pasto en `dq` y la energía >10 Hz son la firma del temblor. Cazamos sus causas una
  por una (ver README): paso de física grueso (`SIMULATE_DT` 0.005 → 0.002 fue el
  culpable principal), obstáculos en la escena, PD congelado entre comandos, timing del
  loop. Con todo arreglado, `himloco` trota con gyro std ~0.2 rad/s y |dq| < 15 rad/s.
- Números de referencia para comparar tus grabaciones (comando 0.5 m/s, piso plano):
  `himloco` (0.18, 0.29, 0.17) rad/s de std de gyro; `robot_lab` (0.45, 0.47, 0.18).
- La deriva de rumbo no se ve acá — está en `rt/sportmodestate` (posición del cuerpo),
  que este script no graba. Ejercicio: extender `02_record_lowstate.py` para grabar
  también ese topic y graficar la trayectoria XY.

**El gap sim2sim residual sigue ahí** (entrenadas en otro motor de física): la solución
de fondo es entrenar la nuestra en MuJoCo con mjlab — el próximo gran hito.